In [1]:
!which pip

~/miniconda3/envs/autopath/bin/pip


In [2]:
from importlib import reload
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import torch

HOME = os.environ['HOME']
DINOV2 = f"{HOME}/dinov2"
sys.path.insert(0, DINOV2)

# MODEL

In [3]:
import autopath.gigaq.dinov2.train
reload(autopath.gigaq.dinov2.train)
from autopath.gigaq.dinov2.train import make_args, make_cfg, SSL

/home/t-9dkarp/dinov2/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/t-9dkarp/dinov2/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/t-9dkarp/dinov2/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")
/home/t-9dkarp/miniconda3/envs/autopath/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-08-01 23:37:05,705	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


 cuInit Failed, error CUDA_ERROR_NO_DEVICE
 cuFile initialization failed


In [4]:
args = make_args([])
cfg = make_cfg(args)
ssl = SSL(cfg, device='cpu')
#ssl.prepare_for_distributed_training()

I20250801 23:37:07 2563888 dinov2 cfg.py:73] git:
  sha: e1277af2ba9496fbadf7aec6eba56e8d882d1e35, status: has uncommitted changes, branch: autoi/gigapath_uq

I20250801 23:37:07 2563888 dinov2 cfg.py:74] config_file: /home/t-9dkarp/autopath/autopath/gigaq/dinov2/ssl.yaml
eval: 
eval_only: False
no_resume: False
opts: ['train.output_dir=/home/t-9dkarp/autopath/notebooks']
output_dir: /home/t-9dkarp/autopath/notebooks
I20250801 23:37:07 2563888 dinov2 cfg.py:33] sqrt scaling learning rate; base: 0.004, new: 0.001
I20250801 23:37:07 2563888 dinov2 cfg.py:40] MODEL:
  WEIGHTS: ''
compute_precision:
  grad_scaler: true
  teacher:
    backbone:
      sharding_strategy: SHARD_GRAD_OP
      mixed_precision:
        param_dtype: fp16
        reduce_dtype: fp16
        buffer_dtype: fp32
    dino_head:
      sharding_strategy: SHARD_GRAD_OP
      mixed_precision:
        param_dtype: fp16
        reduce_dtype: fp16
        buffer_dtype: fp32
    ibot_head:
      sharding_strategy: SHARD_GRAD_OP


# DATASET

In [13]:
import autopath.gigaq.dinov2.augmentations
reload(autopath.gigaq.dinov2.augmentations)
import autopath.gigaq.dinov2.dataset
reload(autopath.gigaq.dinov2.dataset)

<module 'autopath.gigaq.dinov2.dataset' from '/home/t-9dkarp/autopath/autopath/gigaq/dinov2/dataset.py'>

In [14]:
dataset = autopath.gigaq.dinov2.dataset.make_dataset(verbose=False)

I20250801 23:48:08 2563888 dinov2 augmentations.py:80] ###################################
I20250801 23:48:08 2563888 dinov2 augmentations.py:81] Using data augmentation parameters:
I20250801 23:48:08 2563888 dinov2 augmentations.py:82] global_crops_scale: [0.32, 1.0]
I20250801 23:48:08 2563888 dinov2 augmentations.py:83] local_crops_scale: [0.05, 0.32]
I20250801 23:48:08 2563888 dinov2 augmentations.py:84] local_crops_number: 8
I20250801 23:48:08 2563888 dinov2 augmentations.py:85] global_crops_size: 224
I20250801 23:48:08 2563888 dinov2 augmentations.py:86] local_crops_size: 96
I20250801 23:48:08 2563888 dinov2 augmentations.py:87] ###################################


In [17]:
import autopath.gigaq.dinov2.dataloader

In [18]:
dataloader = autopath.gigaq.dinov2.dataloader.make_dataloader(dataset)

I20250801 23:49:47 2563888 dinov2 loaders.py:159] sampler: none
I20250801 23:49:47 2563888 dinov2 loaders.py:206] using PyTorch data loader
I20250801 23:49:47 2563888 dinov2 loaders.py:219] # of batches: 905,834


### EVAL